# 09 — Temporal Fusion Transformer (TFT) for Groundwater Level Forecasting

**Goal:** Multi-horizon prediction of district-level groundwater levels (t+7, t+14, t+30)  
**Output:** q10 / q50 / q90 quantile forecasts per district per horizon  
**Reference:** Lim et al. (2021) — *Temporal Fusion Transformers for Interpretable Multi-horizon Time Series Forecasting*

> **NOTE:** If kernel was restarted due to a crash, that was caused by `pytorch-forecasting` which has been uninstalled.  
> Restart the kernel once and run all cells — it will work fine now.

---
### Training time estimates
| Mode | Config | CPU time/epoch | GPU time/epoch |
|------|--------|---------------|----------------|
| FAST (default) | d=64, enc=60, 1-layer | ~3 min | ~15 sec |
| FULL | d=128, enc=90, 2-layer | ~12 min | ~45 sec |

Set `FAST_MODE = False` only if you have a GPU. For CPU training, `FAST_MODE = True` is recommended (~2-3 hours total).

In [2]:
import sys, warnings, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, str(Path('.').resolve()))
from tft_model import TemporalFusionTransformer, QuantileLoss

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', DEVICE)

if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU — FAST_MODE will be set automatically')

Device: cpu
Running on CPU — FAST_MODE will be set automatically


In [3]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────
# FAST_MODE = True  → CPU-friendly (d=64, enc=60, 1-layer LSTM) ~3 min/epoch
# FAST_MODE = False → full model  (d=128, enc=90, 2-layer LSTM) needs GPU
FAST_MODE = DEVICE.type == 'cpu'   # auto: True on CPU, False on GPU

if FAST_MODE:
    ENCODER_LEN  = 60
    D_MODEL      = 64
    N_HEADS      = 4
    LSTM_LAYERS  = 1
    BATCH_SIZE   = 128
    MAX_EPOCHS   = 80
    PATIENCE     = 12
    print('FAST_MODE (CPU): d_model=64, encoder=60, 1-layer LSTM')
else:
    ENCODER_LEN  = 90
    D_MODEL      = 128
    N_HEADS      = 4
    LSTM_LAYERS  = 2
    BATCH_SIZE   = 64
    MAX_EPOCHS   = 100
    PATIENCE     = 15
    print('FULL MODE (GPU): d_model=128, encoder=90, 2-layer LSTM')

DECODER_LEN    = 30
EVAL_STEPS     = [6, 13, 29]   # 0-indexed → t+7, t+14, t+30
DROPOUT        = 0.1
QUANTILES      = [0.1, 0.5, 0.9]
CAT_EMBED_DIM  = 8
LR             = 1e-3
GRAD_CLIP      = 1.0
TRAIN_END      = '2025-01-01'

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

OUT_DIR   = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)
DATA_PATH = Path('../../Data Preprocessing/clean datasets/master_dataset.csv')

print(f'\nConfig: enc={ENCODER_LEN}, dec={DECODER_LEN}, d={D_MODEL}, heads={N_HEADS}, lstm_layers={LSTM_LAYERS}')
print(f'Train split: up to {TRAIN_END}, val: rest')

FAST_MODE (CPU): d_model=64, encoder=60, 1-layer LSTM

Config: enc=60, dec=30, d=64, heads=4, lstm_layers=1
Train split: up to 2025-01-01, val: rest


## 1. Load & Impute Data

In [4]:
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df = df.sort_values(['district', 'date']).reset_index(drop=True)
print(f'Raw shape: {df.shape}')
print(f'Districts: {df["district"].nunique()}')
print(f'Date range: {df["date"].min().date()} -> {df["date"].max().date()}')

SOIL_COLS     = ['sand_0cm','clay_0cm','silt_0cm','sand_10cm','clay_10cm',
                 'silt_10cm','sand_30cm','clay_30cm','silt_30cm']
SENTINEL_COLS = ['ndvi_mean','ndwi_mean','ndmi_mean','vv_db','vh_db','vh_vv_ratio']
LAG_COLS      = ['gwl_lag_7','gwl_lag_14','gwl_lag_30','gwl_7d_change','gwl_30d_change']

# impute 4 districts missing soil/sentinel with dataset medians
df[SOIL_COLS]     = df[SOIL_COLS].fillna(df[SOIL_COLS].median())
df[SENTINEL_COLS] = df[SENTINEL_COLS].fillna(df[SENTINEL_COLS].median())

# fill lag NaNs at start of each district series
df[LAG_COLS] = df.groupby('district')[LAG_COLS].transform(lambda s: s.fillna(s.median()))

# encode season as integer for model input
df['season_id'] = df['season'].map({'winter':0,'summer':1,'monsoon':2,'post_monsoon':3}).astype(int)

remaining_nans = df.isnull().sum().sum()
print(f'Missing after imputation: {remaining_nans} (should be 0)')
print(f'Total rows: {len(df):,}')

Raw shape: (16785, 42)
Districts: 24
Date range: 2023-03-02 -> 2025-04-22
Missing after imputation: 0 (should be 0)
Total rows: 16,785


## 2. Feature Groups

In [5]:
STATIC_CAT  = ['district_id']   # embedded → learns spatial representation
STATIC_CONT = SOIL_COLS         # 9 soil texture features (never change per district)

PAST_CONT = [                   # 26 features — only known up to today
    'gwl_mean',
    'gwl_min', 'gwl_max', 'station_count',
    'rainfall_mm', 'rainfall_7d', 'rainfall_30d',
    'et_mm', 'soil_moisture', 'net_water_balance',
    'temp_max', 'temp_min', 'temp_mean', 'humidity_mean', 'vpd_approx',
    'gwl_lag_7', 'gwl_lag_14', 'gwl_lag_30',
    'gwl_7d_change', 'gwl_30d_change',
    'ndvi_mean', 'ndwi_mean', 'ndmi_mean',
    'vv_db', 'vh_db', 'vh_vv_ratio',
]

FUTURE_CONT = ['month', 'day_of_year', 'is_monsoon', 'season_id']  # 4 calendar features
TARGET      = 'gwl_mean'

N_DISTRICTS = df['district_id'].nunique()
print(f'Past features   : {len(PAST_CONT)}')
print(f'Future features : {len(FUTURE_CONT)}')
print(f'Static cont     : {len(STATIC_CONT)}')
print(f'Districts       : {N_DISTRICTS}')

Past features   : 26
Future features : 4
Static cont     : 9
Districts       : 24


## 3. Normalisation

In [6]:
train_mask     = df['date'] < TRAIN_END
cols_to_scale  = PAST_CONT + STATIC_CONT + FUTURE_CONT

scaler = StandardScaler()
scaler.fit(df.loc[train_mask, cols_to_scale])

df_norm               = df.copy()
df_norm[cols_to_scale] = scaler.transform(df[cols_to_scale])

target_scaler = StandardScaler()
target_scaler.fit(df.loc[train_mask, [TARGET]])
df_norm[TARGET] = target_scaler.transform(df[[TARGET]])

print(f'Train rows : {train_mask.sum():,}')
print(f'Val rows   : {(~train_mask).sum():,}')
print(f'Target mean (train): {df.loc[train_mask, TARGET].mean():.2f} m')
print(f'Target std  (train): {df.loc[train_mask, TARGET].std():.2f} m')

Train rows : 14,148
Val rows   : 2,637
Target mean (train): -13.53 m
Target std  (train): 15.12 m


## 4. Dataset & DataLoader

In [7]:
class GWLDataset(Dataset):
    """
    Sliding window dataset for TFT.
    Each window: encoder_len past days + decoder_len future days.
    Static features are the same for all rows of a district.
    """
    def __init__(self, df, encoder_len, decoder_len,
                 past_cols, future_cols, static_cat_cols, static_cont_cols,
                 target_col, split='train', train_end=TRAIN_END):
        self.windows = []
        for dist, grp in df.groupby('district'):
            grp = grp.sort_values('date').reset_index(drop=True)
            n   = len(grp)
            for i in range(encoder_len, n - decoder_len + 1):
                anchor = grp.loc[i - 1, 'date']
                if split == 'train' and anchor >= pd.Timestamp(train_end): continue
                if split == 'val'   and anchor <  pd.Timestamp(train_end): continue

                past_df   = grp.iloc[i - encoder_len : i]
                future_df = grp.iloc[i : i + decoder_len]

                pc = past_df[past_cols].values.astype(np.float32)
                fc = future_df[future_cols].values.astype(np.float32)
                sc = grp.loc[0, static_cat_cols].values.astype(np.int64)
                sx = grp.loc[0, static_cont_cols].values.astype(np.float32)
                tg = future_df[target_col].values.astype(np.float32)

                if np.isnan(pc).any() or np.isnan(fc).any() or np.isnan(tg).any():
                    continue

                self.windows.append({
                    'past_cont': pc, 'future_cont': fc,
                    'static_cat': sc, 'static_cont': sx, 'target': tg,
                    'district': dist, 'date': future_df['date'].values[-1],
                })

    def __len__(self): return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        return (
            torch.from_numpy(w['static_cat']),
            torch.from_numpy(w['static_cont']),
            torch.from_numpy(w['past_cont']),
            torch.from_numpy(w['future_cont']),
            torch.from_numpy(w['target']),
        )


print('Building datasets...')
t0 = time.time()
train_ds = GWLDataset(df_norm, ENCODER_LEN, DECODER_LEN,
                      PAST_CONT, FUTURE_CONT, STATIC_CAT, STATIC_CONT, TARGET, split='train')
val_ds   = GWLDataset(df_norm, ENCODER_LEN, DECODER_LEN,
                      PAST_CONT, FUTURE_CONT, STATIC_CAT, STATIC_CONT, TARGET, split='val')
print(f'Done in {time.time()-t0:.1f}s')
print(f'Train windows : {len(train_ds):,}')
print(f'Val windows   : {len(val_ds):,}')

# pin_memory only helps on GPU; on CPU it has no effect and wastes time
pin = DEVICE.type == 'cuda'
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin)
print(f'Batches/epoch : {len(train_loader)}')

# training time estimate
secs_per_batch = 0.9 if FAST_MODE else 3.7   # measured benchmarks
est_mins = secs_per_batch * len(train_loader) / 60
print(f'Estimated time/epoch on CPU: {est_mins:.1f} min (~{est_mins*PATIENCE*2:.0f} min total)')

Building datasets...
Done in 20.1s
Train windows : 12,732
Val windows   : 1,917
Batches/epoch : 100
Estimated time/epoch on CPU: 1.5 min (~36 min total)


## 5. Model Initialisation

In [1]:
model = TemporalFusionTransformer(
    n_static_cat    = len(STATIC_CAT),
    cat_vocab_sizes = [N_DISTRICTS],
    cat_embed_dim   = CAT_EMBED_DIM,
    n_static_cont   = len(STATIC_CONT),
    n_past          = len(PAST_CONT),
    n_future        = len(FUTURE_CONT),
    encoder_len     = ENCODER_LEN,
    decoder_len     = DECODER_LEN,
    d_model         = D_MODEL,
    n_heads         = N_HEADS,
    lstm_layers     = LSTM_LAYERS,
    dropout         = DROPOUT,
    quantiles       = QUANTILES,
).to(DEVICE)

criterion = QuantileLoss(QUANTILES).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6, verbose=True
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'TFT parameters : {n_params:,}')
print(f'Device         : {DEVICE}')
print(f'Mode           : {"FAST (CPU)" if FAST_MODE else "FULL (GPU)"}')

NameError: name 'TemporalFusionTransformer' is not defined

## 6. Training

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None, grad_clip=None):
    training = optimizer is not None
    model.train(training)
    total_loss, n = 0.0, 0
    with torch.set_grad_enabled(training):
        for batch in loader:
            sc, sx, px, fx, y = [b.to(DEVICE) for b in batch]
            preds, _, _ = model(sc, sx, px, fx)
            loss = criterion(preds, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                if grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            total_loss += loss.item()
            n += 1
    return total_loss / n


train_losses, val_losses = [], []
best_val_loss    = float('inf')
patience_counter = 0
best_state       = None
train_start      = time.time()

print(f'Training for up to {MAX_EPOCHS} epochs | early stopping patience={PATIENCE}')
print(f'{len(train_loader)} batches/epoch | batch size={BATCH_SIZE}')
print('-' * 65)

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    tr  = run_epoch(train_loader, model, criterion, optimizer, GRAD_CLIP)
    val = run_epoch(val_loader,   model, criterion)
    scheduler.step(val)
    train_losses.append(tr)
    val_losses.append(val)

    elapsed = time.time() - t0
    total_h = (time.time() - train_start) / 3600
    lr_now  = optimizer.param_groups[0]['lr']
    print(f'Ep {epoch:03d} | train={tr:.4f} val={val:.4f} | lr={lr_now:.1e} | {elapsed:.0f}s | {total_h:.1f}h total')

    if val < best_val_loss:
        best_val_loss    = val
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stop at epoch {epoch}  (best val={best_val_loss:.4f})')
            break

model.load_state_dict(best_state)
torch.save(best_state, OUT_DIR / 'tft_checkpoint.pt')
print(f'\nBest model saved -> outputs/tft_checkpoint.pt')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train')
ax.plot(val_losses,   label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Quantile loss')
ax.set_title('TFT Training Curve')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'tft_training_curve.png', dpi=150)
plt.show()

## 7. Evaluation (RMSE / MAE / MAPE at t+7, t+14, t+30)

In [ ]:
def evaluate(loader, model, eval_steps, dataset):
    model.eval()
    records = []
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            sc, sx, px, fx, y = [b.to(DEVICE) for b in batch]
            preds, _, _ = model(sc, sx, px, fx)   # (B, T_dec, 3)
            B = y.shape[0]
            for step in eval_steps:
                h = step + 1
                act = target_scaler.inverse_transform(y[:,step].cpu().numpy().reshape(-1,1)).flatten()
                q10 = target_scaler.inverse_transform(preds[:,step,0].cpu().numpy().reshape(-1,1)).flatten()
                q50 = target_scaler.inverse_transform(preds[:,step,1].cpu().numpy().reshape(-1,1)).flatten()
                q90 = target_scaler.inverse_transform(preds[:,step,2].cpu().numpy().reshape(-1,1)).flatten()
                for i in range(B):
                    w = dataset.windows[batch_idx * BATCH_SIZE + i]
                    records.append({'district': w['district'], 'date': str(w['date']),
                                    'horizon': f't+{h}', 'actual': float(act[i]),
                                    'q10': float(q10[i]), 'q50': float(q50[i]), 'q90': float(q90[i])})
    return pd.DataFrame(records)


def metrics(df_pred):
    rows = []
    for h, g in df_pred.groupby('horizon'):
        a, p = g['actual'].values, g['q50'].values
        rows.append({'Horizon': h,
                     'RMSE (m)': round(np.sqrt(np.mean((a-p)**2)), 4),
                     'MAE (m)':  round(np.mean(np.abs(a-p)), 4),
                     'MAPE (%)': round(np.mean(np.abs((a-p)/(np.abs(a)+1e-8)))*100, 2)})
    return pd.DataFrame(rows).set_index('Horizon')


print('Evaluating...')
val_preds   = evaluate(val_loader, model, EVAL_STEPS, val_ds)
val_metrics = metrics(val_preds)

print('\n=== TFT Val Metrics (q50 = median forecast) ===')
print(val_metrics.to_string())

val_preds.to_csv(OUT_DIR / 'tft_predictions.csv', index=False)
val_metrics.to_csv(OUT_DIR / 'tft_metrics.csv')

In [ ]:
# per-district breakdown at t+30
t30 = val_preds[val_preds['horizon'] == 't+30']
rows = []
for dist, g in t30.groupby('district'):
    a, p = g['actual'].values, g['q50'].values
    rows.append({'District': dist,
                 'RMSE': round(np.sqrt(np.mean((a-p)**2)), 3),
                 'MAE':  round(np.mean(np.abs(a-p)), 3),
                 'MAPE': round(np.mean(np.abs((a-p)/(np.abs(a)+1e-8)))*100, 2)})
print('\n=== Per-district MAPE at t+30 ===')
print(pd.DataFrame(rows).sort_values('MAPE').to_string(index=False))

## 8. Quantile Interval Plots

In [ ]:
def plot_quantile(df_pred, district, horizon='t+30', n_days=90):
    sub = df_pred[(df_pred['district']==district) & (df_pred['horizon']==horizon)]\
          .sort_values('date').tail(n_days)
    if sub.empty: return
    x = range(len(sub))
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(x, sub['q10'], sub['q90'], alpha=0.2, color='steelblue', label='80% interval (q10-q90)')
    ax.plot(x, sub['q50'],    color='steelblue', lw=1.5, label='q50 forecast')
    ax.plot(x, sub['actual'], color='black',    lw=1.5, ls='--', label='Actual GWL')
    ax.set_title(f'{district} — {horizon}')
    ax.set_xlabel('Day'); ax.set_ylabel('GWL (m below ground)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'quantile_{district.lower().replace(" ","_")}_{horizon}.png', dpi=150)
    plt.show()

for dist in val_preds['district'].unique()[:4]:
    plot_quantile(val_preds, dist, 't+30')

## 9. VSN Feature Importance

In [ ]:
model.eval()
pw, fw, sw = [], [], []
with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= 30: break
        sc, sx, px, fx, _ = [b.to(DEVICE) for b in batch]
        _, vsn_w, _ = model(sc, sx, px, fx)
        pw.append(vsn_w['past'].cpu().mean(dim=(0,1)))
        fw.append(vsn_w['future'].cpu().mean(dim=(0,1)))
        sw.append(vsn_w['static'].cpu().mean(dim=0))

past_imp = torch.stack(pw).mean(0).numpy()
fut_imp  = torch.stack(fw).mean(0).numpy()
stat_imp = torch.stack(sw).mean(0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax = axes[0]
idx = np.argsort(past_imp)[::-1]
ax.barh([PAST_CONT[i] for i in idx], past_imp[idx], color='steelblue')
ax.set_title('Past Feature Importance (VSN)'); ax.set_xlabel('Mean weight'); ax.invert_yaxis()

ax = axes[1]
idx = np.argsort(fut_imp)[::-1]
ax.barh([FUTURE_CONT[i] for i in idx], fut_imp[idx], color='coral')
ax.set_title('Future Feature Importance (VSN)'); ax.set_xlabel('Mean weight'); ax.invert_yaxis()

ax = axes[2]
snames = ['district_emb'] + STATIC_CONT
idx = np.argsort(stat_imp)[::-1]
ax.barh([snames[i] for i in idx], stat_imp[idx], color='seagreen')
ax.set_title('Static Feature Importance (VSN)'); ax.set_xlabel('Mean weight'); ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUT_DIR / 'tft_vsn_importance.png', dpi=150)
plt.show()
print('Saved -> outputs/tft_vsn_importance.png')

## 10. Attention Heatmap

In [ ]:
model.eval()
n_samples = 5
batch = next(iter(val_loader))
sc, sx, px, fx, _ = [b[:n_samples].to(DEVICE) for b in batch]
with torch.no_grad():
    _, _, attn_w = model(sc, sx, px, fx)   # (B, T_dec, T_enc+T_dec)
mean_attn = attn_w.mean(dim=1).cpu().numpy()

fig, axes = plt.subplots(n_samples, 1, figsize=(14, 3*n_samples))
for i in range(n_samples):
    ax = axes[i]
    ax.bar(range(mean_attn.shape[1]), mean_attn[i], width=1.0, color='steelblue', alpha=0.7)
    ax.axvline(x=ENCODER_LEN-0.5, color='red', ls='--', lw=1.5, label='Past | Future')
    ax.set_ylabel('Attn weight')
    ax.set_title(f'Sample {i+1}')
    if i == 0: ax.legend()
axes[-1].set_xlabel(f'Timestep  (0-{ENCODER_LEN-1}: past history | {ENCODER_LEN}-{ENCODER_LEN+DECODER_LEN-1}: future)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'tft_attention.png', dpi=150)
plt.show()

## 11. Comparison with Baselines

In [ ]:
baseline_path = Path('../baselines/outputs/multihorizon_summary.csv')
if baseline_path.exists():
    base = pd.read_csv(baseline_path)
    tft_rows = []
    for h in ['t+7','t+14','t+30']:
        if h in val_metrics.index:
            r = val_metrics.loc[h]
            tft_rows.append({'Model':'TFT','Horizon':h,
                             'RMSE':r['RMSE (m)'],'MAE':r['MAE (m)'],'MAPE':r['MAPE (%)']})
    combined = pd.concat([base, pd.DataFrame(tft_rows)], ignore_index=True)
    combined.to_csv(OUT_DIR / 'comparison_with_tft.csv', index=False)

    pivot = combined.pivot_table(index='Model', columns='Horizon', values='MAPE')
    print('\n=== MAPE (%) — All Models ===')
    print(pivot.round(2).to_string())

    # bar chart
    ax = pivot.plot(kind='bar', figsize=(10,5), rot=0)
    ax.set_ylabel('MAPE (%)'); ax.set_title('MAPE Comparison: Baselines vs TFT')
    ax.legend(title='Horizon'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'mape_comparison.png', dpi=150)
    plt.show()
else:
    print('Baseline file not found. TFT metrics:')
    print(val_metrics.to_string())

## 12. Save Summary

In [ ]:
summary = {
    'model': 'TFT', 'fast_mode': FAST_MODE, 'device': str(DEVICE),
    'encoder_len': ENCODER_LEN, 'decoder_len': DECODER_LEN,
    'd_model': D_MODEL, 'n_heads': N_HEADS, 'lstm_layers': LSTM_LAYERS,
    'dropout': DROPOUT, 'quantiles': QUANTILES, 'n_params': n_params,
    'best_val_loss': round(best_val_loss, 6),
    'epochs_trained': len(train_losses),
    'metrics': val_metrics.reset_index().to_dict('records'),
}
with open(OUT_DIR / 'tft_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('=== Final TFT Summary ===')
for k, v in summary.items():
    if k != 'metrics': print(f'  {k}: {v}')
print()
print(val_metrics.to_string())
print('\nOutputs saved to Models/TFT/outputs/')